# 02 · Historical financial analysis: Disney / 21st Century Fox

Stage 2 of the case study. Every number comes from the verified data layer (`data/processed/historical_financials.csv`,
built and checked in `01_data_collection.ipynb`). Nothing was re-collected or typed in by hand.

* `src/build_analysis.py` → `financial_ratios.csv`, `segment_analysis.csv`. Each row lists its inputs as
  `name=value [record_id; source_id p.page]`.
* `src/make_charts.py` → `docs/figures/*.png`.
* Written findings: `docs/historical_analysis_report.md`.

**Labels:** `reported` (unchanged from the dataset) · `derived` (arithmetic on reported figures, formula given)
· `adjusted` (derived and excluding restructuring/impairment) · `not_available` (cannot be computed, with the reason).
No estimates. **No valuation** (no DCF, WACC, comparables or synergies).

**Fiscal years are not aligned.** Disney's year ends on the Saturday nearest 30 Sep (FY2015 = 53 weeks); 21CF's
and RemainCo's end on 30 Jun. "FY2017" therefore means Oct-16 to Sep-17 for Disney and Jul-16 to Jun-17 for 21CF.

In [ ]:
from pathlib import Path
import subprocess, sys
import pandas as pd
from IPython.display import Image, display

pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90); pd.set_option("display.max_rows", 200)
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
PROC, FIG = ROOT / "data" / "processed", ROOT / "docs" / "figures"
for script in ("build_analysis.py", "make_charts.py"):
    res = subprocess.run([sys.executable, str(ROOT / "src" / script)], cwd=ROOT, capture_output=True, text=True, encoding="utf-8")
    print(res.stdout.strip()); assert res.returncode == 0, res.stderr
r = pd.read_csv(PROC / "financial_ratios.csv"); r["v"] = pd.to_numeric(r.value, errors="coerce")
s = pd.read_csv(PROC / "segment_analysis.csv"); s["v"] = pd.to_numeric(s.value, errors="coerce")

def table(metrics, entities=("DIS", "21CF"), pct=()):
    q = r[r.metric.isin(metrics) & r.entity.isin(entities)]
    t = q.pivot_table(index="metric", columns=["entity", "fiscal_period"], values="v", aggfunc="first").reindex(metrics)
    return t.apply(lambda row: row.map(lambda v: f"{v:.1%}" if row.name in pct and pd.notna(v) else (f"{v:,.2f}" if pd.notna(v) else "")), axis=1)

## 1. Historical performance
### Revenue growth

In [ ]:
display(table(["revenue", "revenue_growth_yoy"], pct=["revenue_growth_yoy"]))
r[r.metric.str.startswith("revenue_cagr")][["entity", "metric", "value", "formula", "inputs"]]

In [ ]:
display(Image(FIG / "01_revenue_trends.png"))

### Margins, EBITDA, tax and returns
EBIT is **derived** as operating income before equity income, other income/expense and interest. Neither company
reports this subtotal. EBITDA adds back income-statement D&A. Gross margin is not comparable between the companies:
Disney's cost of services and products excludes D&A, while 21CF's single "operating expenses" line has a different scope.

In [ ]:
m = ["gross_margin_excl_da", "ebitda", "ebitda_margin", "ebitda_excl_restructuring_margin", "ebit", "ebit_margin",
     "net_margin", "effective_tax_rate", "roa_avg_assets", "roe_avg_equity"]
table(m, pct=[x for x in m if x not in ("ebitda", "ebit")])

In [ ]:
display(Image(FIG / "02_ebit_ebitda_trends.png")); display(Image(FIG / "03_profitability_margins.png"))

## 2. Segments
Disney reports segment **operating income** (after D&A); 21CF reports **Segment OIBDA** (before D&A).
**21CF segments describe the consolidated pre-separation company, not the business Disney acquired.**
No RemainCo segment data exists in the sources (written as `not_available` rows).

In [ ]:
sp = s[s.metric.isin(["segment_revenue", "revenue_share", "segment_margin"]) & s.entity.isin(["DIS", "21CF"])]
sp.pivot_table(index=["entity", "segment", "metric"], columns="fiscal_period", values="v", aggfunc="first", sort=False).round(3)

In [ ]:
display(Image(FIG / "04_segment_revenue_composition.png")); display(Image(FIG / "05_segment_margins.png"))

## 3. Balance sheet and cash flow

In [ ]:
table(["cash", "total_borrowings", "net_debt", "total_equity", "working_capital", "working_capital_excl_cash_debt",
       "debt_to_equity", "debt_to_capital", "net_debt_to_ebitda", "ebit_interest_cover",
       "cfo", "capex", "fcf_cfo_less_capex", "capex_to_revenue", "fcf_margin", "cfo_to_ebitda", "fcf_to_net_income"],
      pct=["capex_to_revenue", "fcf_margin"])

In [ ]:
display(Image(FIG / "06_debt_and_cash.png")); display(Image(FIG / "07_cash_flow_trends.png"))

## 4. RemainCo, the acquired business
Pro forma figures (S-X Article 11). FY2015 and 6M Dec-17 come from the 2018 joint proxy; FY2016-FY2018 and Q1 FY2019
from the March 2019 8-K/A (closing perimeter); balance sheets at Mar-18 (proxy) and Dec-18 (8-K/A).

In [ ]:
rc = r[r.entity == "RemainCo"]
rc.pivot_table(index="metric", columns="fiscal_period", values="v", aggfunc="first")[
    ["FY2015", "FY2016", "FY2017", "6M_DEC2017", "FY2018", "Q1_FY2019", "MAR2018", "DEC2018"]].round(3)

In [ ]:
# explicitly missing items, with reasons
r[r.value_type == "not_available"][["entity", "fiscal_period", "metric", "notes"]].drop_duplicates(["metric", "notes"])

## 5. Traceability check
Each derived value can be recomputed from the inputs it cites. As a spot check, recompute three of them from
`historical_financials.csv`.

In [ ]:
hf = pd.read_csv(PROC / "historical_financials.csv", low_memory=False).set_index("record_id")
import re
def recompute(entity, period, metric):
    row = r[(r.entity == entity) & (r.fiscal_period == period) & (r.metric == metric)].iloc[0]
    ids = re.findall(r"\[(HF\d+);", row.inputs)
    return row.value, row.formula, {i: hf.loc[i, "value"] for i in ids}
for args in [("DIS", "FY2017", "ebit"), ("21CF", "FY2017", "ebit"), ("RemainCo", "FY2017", "ebit")]:
    v, f, ins = recompute(*args)
    assert abs(sum(ins.values()) - float(v)) < 0.01, (args, v, ins)
    print(args, v, "=", f, "=", " + ".join(f"{x:g}" for x in ins.values()))